## Examples 1-5: the filesystem registry, the in-memory backend, the URL layer, chained caching and local files with compression

Every example runs offline, against fsspec's in-process `memory` backend or a temporary local directory.

In [ ]:
# Example 1 — the filesystem registry: a protocol string resolved to a filesystem class.
#
# get_filesystem_class resolves a protocol string to its filesystem class; fsspec.filesystem()
# constructs it and caches the instance. `memory` and `file` are both in-process, so the cell
# needs no network and no credentials.
import fsspec
from fsspec.registry import get_filesystem_class

for protocol, expected_class_name in (("memory", "MemoryFileSystem"), ("file", "LocalFileSystem")):
    resolved = get_filesystem_class(protocol)
    assert isinstance(resolved, type), (protocol, resolved)
    assert issubclass(resolved, fsspec.AbstractFileSystem), f"{protocol!r} -> {resolved!r}, not an AbstractFileSystem"
    assert resolved.__name__ == expected_class_name, (protocol, resolved.__name__)
    print(f"registry resolved {protocol!r} -> {resolved.__module__}.{resolved.__name__}")

# fsspec.filesystem() constructs the class and returns the same cached instance for the same
# class and arguments.
filesystem = fsspec.filesystem("memory")
assert isinstance(filesystem, fsspec.AbstractFileSystem), type(filesystem)
assert filesystem is fsspec.filesystem("memory"), "fsspec instance cache handed back a 2nd object"
print("constructed:", type(filesystem).__name__, "| instance cache: hit")

In [ ]:
# Example 2 — a write / read / list / delete round trip through the in-process `memory` backend.
#
# pipe_file / cat_file / ls / info / rm / exists are AbstractFileSystem methods shared by every backend.
# `MemoryFileSystem.store` is a class attribute shared by the whole kernel, so the paths sit under a
# cell-unique prefix and the listing is checked by containment rather than compared exactly.
import fsspec

filesystem = fsspec.filesystem("memory")
root = "/fsspec-nb-roundtrip"
payload = b"fsspec round trip \x00\xff bytes"
filesystem.pipe_file(f"{root}/a.bin", payload)
filesystem.pipe_file(f"{root}/b.bin", b"second")

assert filesystem.cat_file(f"{root}/a.bin") == payload, "memory backend returned different bytes"
assert filesystem.info(f"{root}/a.bin")["size"] == len(payload), filesystem.info(f"{root}/a.bin")
listed = set(filesystem.ls(root, detail=False))
assert {f"{root}/a.bin", f"{root}/b.bin"} <= listed, listed
assert filesystem.exists(f"{root}/a.bin"), listed

filesystem.rm(f"{root}/a.bin")
assert not filesystem.exists(f"{root}/a.bin"), "rm left the key readable"
assert filesystem.cat_file(f"{root}/b.bin") == b"second", "rm collaterally damaged a sibling"
filesystem.rm(f"{root}/b.bin")
print(f"round trip: {len(payload)} bytes written, read back identical, listed, deleted")

In [ ]:
# Example 3 — the URL layer (`fsspec.core`): fsspec.open() writes and reads through an OpenFile,
# url_to_fs splits a URL into a filesystem and a path, and open_files expands a glob.
import fsspec

base = "memory://fsspec-nb-url"
for name, content in (("one.txt", b"url-layer-one"), ("two.txt", b"url-layer-two")):
    with fsspec.open(f"{base}/{name}", "wb") as handle:
        handle.write(content)

with fsspec.open(f"{base}/one.txt", "rb") as handle:
    assert handle.read() == b"url-layer-one", "fsspec.open() did not read back what it wrote"

resolved_filesystem, resolved_path = fsspec.core.url_to_fs(f"{base}/one.txt")
assert isinstance(resolved_filesystem, fsspec.AbstractFileSystem), type(resolved_filesystem)
assert resolved_path.endswith("/one.txt") and "memory:" not in resolved_path, resolved_path

expanded = fsspec.open_files(f"{base}/*.txt", "rb")
assert len(expanded) == 2, f"glob expanded to {len(expanded)} file(s): {[h.path for h in expanded]}"
assert sorted(h.open().read() for h in expanded) == [b"url-layer-one", b"url-layer-two"], expanded
resolved_filesystem.rm("/fsspec-nb-url", recursive=True)
print("url layer:", len(expanded), "files expanded from one glob, bytes round-tripped")

In [ ]:
# Example 4 — a chained `filecache::` URL: a caching filesystem wrapped around the local one,
# built from a single string. The check is that a blob in the cache directory holds the source
# bytes. Local file in, local cache out: no network, no credentials.
import os
import tempfile

import fsspec

source_bytes = b"fsspec chained cache payload"
with tempfile.TemporaryDirectory() as scratch_path:
    source_path = os.path.join(scratch_path, "source.bin")
    cache_path = os.path.join(scratch_path, "cache")
    with open(source_path, "wb") as handle:
        handle.write(source_bytes)
    with fsspec.open(f"filecache::file://{source_path}", "rb",
                     filecache={"cache_storage": cache_path}) as handle:
        assert handle.read() == source_bytes, "chained `filecache::` URL returned different bytes"
    cached_names = [n for n in os.listdir(cache_path)
                    if os.path.isfile(os.path.join(cache_path, n))
                    and open(os.path.join(cache_path, n), "rb").read() == source_bytes]
    assert cached_names, f"no blob under {cache_path} holds the source bytes: {os.listdir(cache_path)}"
print("chained URL cached the payload:", len(source_bytes), "bytes, blob verified byte-identical")

In [ ]:
# Example 5 — the local filesystem and transparent compression on a temporary directory.
import gzip
import os
import tempfile

import fsspec
from fsspec.implementations.local import LocalFileSystem

text = "line one\nline two\n"
with tempfile.TemporaryDirectory() as scratch_path:
    local = fsspec.filesystem("file")
    assert isinstance(local, LocalFileSystem), type(local)

    plain_path = os.path.join(scratch_path, "plain.txt")
    local.pipe_file(plain_path, b"plain bytes")
    os.makedirs(os.path.join(scratch_path, "sub"))
    listed = {os.path.basename(p) for p in local.ls(scratch_path, detail=False)}
    assert listed == {"plain.txt", "sub"}, listed
    info = local.info(plain_path)
    assert info["type"] == "file" and info["size"] == len(b"plain bytes"), info
    assert local.info(os.path.join(scratch_path, "sub"))["type"] == "directory"

    # compression="gzip" wraps the file object, so the bytes on disk are a gzip stream.
    gz_path = os.path.join(scratch_path, "data.txt.gz")
    with fsspec.open(gz_path, "wt", compression="gzip") as handle:
        handle.write(text)
    with open(gz_path, "rb") as raw:
        assert raw.read(2) == b"\x1f\x8b", "not a gzip stream"
    with gzip.open(gz_path, "rt") as handle:
        assert handle.read() == text, "gzip module read different text"
    # compression="infer" picks the codec from the .gz suffix.
    with fsspec.open(gz_path, "rt", compression="infer") as handle:
        assert handle.read() == text
print("local ls/info and gzip round trip OK")